# Feature Store

## Installation des librairies

Pour cette démo la librairie *Databricks Feature Engineering (ou Feature Store)* est nécéssaire pour la création des tables, chargement des datasets et la publication de **features**.

In [0]:
%pip install databricks-feature-engineering

après l'installation il faut redémarrer le kernel python avec :

In [0]:
dbutils.library.restartPython()

## Mise en place

In [0]:
from databricks.feature_store import FeatureStoreClient

fs = FeatureStoreClient()

catalog_name = "demo_" + spark.sql("SELECT current_user()").collect()[0][0].split("@")[0]
silver_schema = "silver"
gold_schema = "gold"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE {silver_schema}")

## Création et MAJ de features


**Objectif** : Construire des features au niveau client à partir des tables raffinées (`refined_orders` et `refined_customer`) : 
- `total_orders` : nombre total de commandes par client
- `avg_order_value` : prix moyen par commande
- `total_spending` : montant total dépensé
- `market_segment` : une colonne catégorielle provenant de la dimension client





In [0]:
from pyspark.sql.functions import col, count, avg, sum, current_timestamp

orders_df = spark.sql("SELECT * FROM refined_orders")
customers_df = spark.sql("SELECT * FROM refined_customer")

base_features_df = (
    orders_df.groupBy("customer_id")
    .agg(
        count("*").alias("total_orders"),
        avg("total_price").alias("avg_order_value"),
        sum("total_price").alias("total_spending")
    )
    .join(
        customers_df.select("customer_id", "market_segment"),
        on="customer_id",
        how="inner"
    )
    .withColumn("feature_update_ts", current_timestamp())
)

display(base_features_df.limit(5))

### Création d'une table de features

On va créer un table *customer_features* dans le schéma `gold`.

In [0]:
feature_table_name = f"{catalog_name}.{gold_schema}.customer_features"

try:
    fs.create_table(
        name=feature_table_name,
        primary_keys=["customer_id"],
        schema=base_features_df.schema,
        description="Customer features from refined silver tables",
    )
    print(f"Table {feature_table_name} créée.")
except Exception as e:
    print(f"Echec de la création de la table {feature_table_name}")

fs.write_table(
    name=feature_table_name,
    df=base_features_df,
    mode="merge"
)

print(f"La table {feature_table_name} a été mise à jour")

## Entrainement d'un model avec le Feature Store
1. Créez un label simple : les clients ayant dépensé plus qu'un certain seuil sont considérés comme "dépensier" ou "high spender".
2. Recherchez les mêmes features via FeatureLookup pour l'entraînement
3. Entrainer un model de régression logistique basique

In [0]:
from pyspark.sql.functions import when

threshold = 20000.0

labeled_df = (
    base_features_df
    .withColumn("label", when(col("total_spending") > threshold, 1).otherwise(0))
    .select("customer_id", "total_orders", "avg_order_value", "total_spending", "market_segment", "label")
)

display(labeled_df.limit(5))

### Création d'un jeu d'entrainement avec `FeatureLookups`

In [0]:
from databricks.feature_store import FeatureLookup

feature_lookup = FeatureLookup(
    table_name=feature_table_name,
    feature_names=["total_orders", "avg_order_value", "total_spending", "market_segment"],
    lookup_key="customer_id"
)
labeled_df_clean = labeled_df.drop("total_orders", "avg_order_value", "total_spending", "market_segment")

training_set = fs.create_training_set(
    labeled_df_clean,
    feature_lookups=[feature_lookup],
    label="label",
    exclude_columns=["feature_update_ts"]
)

training_df = training_set.load_df()
display(training_df.limit(5))

### Entrainement d'un model de régression logistique

On va transformer une colonne catégorielle (market_segment) en valeur numérique, véctoriser toutes les features et entrainer un classificateur binaire.

In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

indexer = StringIndexer(inputCol="market_segment", outputCol="market_segment_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["market_segment_idx"], outputCols=["market_segment_vec"])

assembler = VectorAssembler(
    inputCols=["total_orders", "avg_order_value", "total_spending", "market_segment_vec"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)

pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])
model = pipeline.fit(training_df)

print("Model training complete")

### inférence par lot en utilisant le Feature Store

On à un nouveau client (ou déjà existant) et on veut prédire si il est *High spender* / *dépensier*

1. Création d'un dataframe d'id de client
2. FeatureLookup
3. Génération de prédictions avec la pipeline entrainée

In [0]:
from pyspark.sql.functions import lit

sample_customers = (
    training_df.select("customer_id")
    .limit(5)
    .withColumn("batch_inference_example", lit(True))
)

display(sample_customers)

### Récupération des features et scores

In [0]:
inference_lookup = FeatureLookup(
    table_name=feature_table_name,
    feature_names=["total_orders", "avg_order_value", "total_spending", "market_segment"],
    lookup_key="customer_id"
)

inference_set = fs.create_training_set(
    df=sample_customers,
    feature_lookups=[inference_lookup],
    label=None
)

inference_df = inference_set.load_df()
predictions = model.transform(inference_df)
display(predictions.select("customer_id", "prediction", "probability"))